# CNN for Artist Classification — Sumanth

Barebones CNN pipeline

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

ModuleNotFoundError: No module named 'torch'

## Configuration

These are the configs I am using for writing the barebones version right now. 64x64 image size is way too small right now, but I did it for speed. Also, the sample frac is the ratio of images I am training on right now. Set it to 1 to train on all. Also def increase the number of epochs.

In [3]:
IMG_SIZE = 64        # resize all images to IMG_SIZE x IMG_SIZE. TODO: 212x212
BATCH_SIZE = 64
NUM_EPOCHS = 2      # TODO: add more epochs
LEARNING_RATE = 1e-3
SAMPLE_FRAC = 0.11       # set < 1.0 to train on a random subset (e.g. 0.25 for 25%). TODO: run on entire set!
NUM_CLASSES = 20         # top 20 artists
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

## Load Splits

train 8K, val 1K, test 1K

In [4]:
# # UNCOMMENT THIS IF YOU'RE HAVING DISK SPACE ISSUES
# from datasets import disable_caching
# disable_caching()  # prevents HF from writing ~4GB Arrow cache to disk

raw = load_dataset("parquet", data_files={
    "train": "train.parquet",
    "val":   "val.parquet",
    "test":  "test.parquet",
}, keep_in_memory=True) # there may be a better way to do this, but this writes to memory


train_hf = raw["train"]
val_hf   = raw["val"]
test_hf  = raw["test"]

if SAMPLE_FRAC < 1.0:
    n_train = max(1, int(len(train_hf) * SAMPLE_FRAC))
    n_val   = max(1, int(len(val_hf)   * SAMPLE_FRAC))
    train_hf = train_hf.shuffle(seed=SEED).select(range(n_train))
    val_hf   = val_hf.shuffle(seed=SEED).select(range(n_val))

print(f"Train: {len(train_hf)}  |  Val: {len(val_hf)}  |  Test: {len(test_hf)}")

# Build artist → label mapping
artist_names = sorted(set(train_hf["artist"]))
artist2idx = {name: i for i, name in enumerate(artist_names)}
idx2artist = {i: name for name, i in artist2idx.items()}
print(f"Classes ({len(artist_names)}): {artist_names[:5]} ...")

FileNotFoundError: Unable to find 'C:/Users/Josh Hall/Documents/GitHub/487finalproj\train.parquet'

## Dataset & DataLoaders

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class ArtDataset(Dataset):
    """Wraps a HuggingFace dataset split into a PyTorch Dataset."""

    def __init__(self, hf_dataset, artist2idx, transform=None):
        self.hf = hf_dataset
        self.artist2idx = artist2idx
        self.transform = transform

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, idx):
        item = self.hf[idx]
        img = item["image"].convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = self.artist2idx[item["artist"]]
        return img, label


train_ds = ArtDataset(train_hf, artist2idx, train_transform)
val_ds   = ArtDataset(val_hf,   artist2idx, eval_transform)
test_ds  = ArtDataset(test_hf,  artist2idx, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches per epoch — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

## CNN Architecture

For now, just 3 block CNN
Each block: Conv → BatchNorm → ReLU → MaxPool

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # 3 conv blocks: 3→32→64→128, each halves spatial dim via MaxPool
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64→32

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32→16

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16→8
        )
        # After 3 pools of a 64×64 input → 8×8 feature maps
        flat_dim = 128 * (IMG_SIZE // 8) * (IMG_SIZE // 8)

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(flat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc  = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{NUM_EPOCHS}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"],   label="val")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(history["train_acc"], label="train")
ax2.plot(history["val_acc"],   label="val")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()

plt.tight_layout()
plt.show()

## Test Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

test_acc = (all_preds == all_labels).float().mean().item()
print(f"Test Accuracy: {test_acc:.4f}")

for cls_idx in range(NUM_CLASSES):
    mask = all_labels == cls_idx
    if mask.sum() == 0:
        continue
    cls_acc = (all_preds[mask] == all_labels[mask]).float().mean().item()
    print(f"  Artist {str(idx2artist[cls_idx]):>5s}  acc={cls_acc:.3f}  (n={mask.sum().item()})")